# Exploded Spike Source Re-generation

In [2]:
import sys
import os
import pandas as pd
import numpy as np
%load_ext autoreload
%autoreload 2

In [4]:
from compile.compile_posthoc_manually_sorted import compile_data
from data_access.cache_utils import ExplodedSpikeCacheManager

mgr = ExplodedSpikeCacheManager()          # monkey="Cortana", subdir="exploded_spike_cache"

sessions = [
    ("2023-09-26", 1),
    ("2023-11-25", 1),
]
import datetime as dt

for session_date, round_no in sessions:
    session_day = dt.date.fromisoformat(session_date)

    compile_data(
        experiment_name=f"{session_day:%y%m%d}_round{round_no}",
        day=session_day,
    )

    mgr.load_or_compute(
        session_date,
        round_no,
        force_recompute=True,
    )

working on 0 out of 1085
working on 1 out of 1085
working on 2 out of 1085
working on 3 out of 1085
working on 4 out of 1085
working on 5 out of 1085
working on 6 out of 1085
working on 7 out of 1085
working on 8 out of 1085
working on 9 out of 1085
working on 10 out of 1085
working on 11 out of 1085
working on 12 out of 1085
working on 13 out of 1085
working on 14 out of 1085
working on 15 out of 1085
working on 16 out of 1085
working on 17 out of 1085
working on 18 out of 1085
working on 19 out of 1085
working on 20 out of 1085
working on 21 out of 1085
working on 22 out of 1085
working on 23 out of 1085
working on 24 out of 1085
working on 25 out of 1085
working on 26 out of 1085
working on 27 out of 1085
working on 28 out of 1085
working on 29 out of 1085
working on 30 out of 1085
working on 31 out of 1085
working on 32 out of 1085
working on 33 out of 1085
working on 34 out of 1085
working on 35 out of 1085
working on 36 out of 1085
working on 37 out of 1085
working on 38 out of 1

In [2]:
import datetime as dt, re
from pathlib import Path

from analyses.data_readers.recording_metadata_reader import RecordingMetadataReader
from compile.compile_common import INTAN_BASE_PATH
from compile.compile_posthoc_manually_sorted import compile_data
from data_access.cache_utils import ExplodedSpikeCacheManager
from project_util import PROJECT_BASE_PATH, SUBJECT_MONKEY

# ---- config ---------------------------------------------------------------
CACHE_DIR = Path(PROJECT_BASE_PATH) / SUBJECT_MONKEY / "exploded_spike_cache_gitrecovered"
DRY_RUN   = False     # True = print the plan, touch nothing
# ---------------------------------------------------------------------------

SESSION_RE = re.compile(r"(\d{4}-\d{2}-\d{2})_round_(\d+)\.pkl$")
sessions = []
for p in sorted(CACHE_DIR.glob("*.pkl")):
    m = SESSION_RE.search(p.name)
    if m:
        sessions.append((m.group(1), int(m.group(2))))
print(f"{len(sessions)} session(s) found in {CACHE_DIR}\n")

reader = RecordingMetadataReader()
mgr = ExplodedSpikeCacheManager()
compiled, reused, unsorted, failed = [], [], [], []

for date, round_no in sessions:
    tag = f"{date} round {round_no}"
    try:
        folder = reader.get_intan_folder_name_for_specific_round(date, round_no)
        round_dir = Path(INTAN_BASE_PATH) / SUBJECT_MONKEY / date / folder

        has_sorted   = (round_dir / "sorted_spikes.pkl").exists()
        has_compiled = (round_dir / "compiled.pkl").exists()

        if not has_sorted:
            unsorted.append(tag); action = "no sorted_spikes.pkl - compiled.pkl not needed"
        elif has_compiled:
            reused.append(tag);   action = "reusing existing compiled.pkl"
        else:
            action = "compiling compiled.pkl"

        print(f"[{tag}] {action}")
        if DRY_RUN:
            continue

        if has_sorted and not has_compiled:
            compile_data(experiment_name=folder, day=dt.date.fromisoformat(date))
            compiled.append(tag)

        df = mgr.load_or_compute(date, round_no, force_recompute=True)
        units = sorted({c for c in df["Channel"].astype(str).unique() if "_Unit" in c})
        print(f"    rebuilt: {df['NeuronID'].nunique()} neurons, "
              f"{df['TaskField'].nunique()} trials, {len(units)} sorted unit(s)")

    except Exception as exc:
        failed.append((tag, f"{type(exc).__name__}: {exc}"))
        print(f"    FAILED - {type(exc).__name__}: {exc}")

print("\n" + "=" * 60)
print(f"regenerated      : {len(sessions) - len(failed)}/{len(sessions)}")
print(f"  compiled first : {len(compiled)}")
print(f"  reused compiled: {len(reused)}")
print(f"  never sorted   : {len(unsorted)}")
if failed:
    print(f"failed           : {len(failed)}")
    for tag, err in failed:
        print(f"  {tag}: {err}")

63 session(s) found in /home/connorlab/Documents/GitHub/Julie/Cortana/exploded_spike_cache_gitrecovered

[2023-09-26 round 1] reusing existing compiled.pkl
[Cache] Using file: /home/connorlab/Documents/GitHub/Julie/Cortana/exploded_spike_cache/2023-09-26_round_1.pkl
    rebuilt: 32 neurons, 350 trials, 19 sorted unit(s)
[2023-09-26 round 2] reusing existing compiled.pkl
[Cache] Using file: /home/connorlab/Documents/GitHub/Julie/Cortana/exploded_spike_cache/2023-09-26_round_2.pkl
    rebuilt: 26 neurons, 360 trials, 5 sorted unit(s)
[2023-09-26 round 3] reusing existing compiled.pkl
[Cache] Using file: /home/connorlab/Documents/GitHub/Julie/Cortana/exploded_spike_cache/2023-09-26_round_3.pkl
    rebuilt: 32 neurons, 338 trials, 18 sorted unit(s)
[2023-09-28 round 1] reusing existing compiled.pkl
[Cache] Using file: /home/connorlab/Documents/GitHub/Julie/Cortana/exploded_spike_cache/2023-09-28_round_1.pkl
    rebuilt: 22 neurons, 340 trials, 2 sorted unit(s)
[2023-09-29 round 2] reusing 